# 1. Prepare the data and four embeddings

Run the notebooks in order, in the **same folder**, using Python 3.10 or newer:

1. **01_data_and_embeddings.ipynb**: inspect the supplied splits; prepare Word2Vec 50d/100d and GloVe 50d/100d.
2. **02_feedforward_experiments.ipynb**: compare embeddings, dimensions, and three hidden-layer configurations (12 experiments).
3. **03_lstm_experiments.ipynb**: compare embeddings, dimensions, and three LSTM configurations (12 experiments).
4. **04_evaluation_and_comparison.ipynb**: compare validation scores; evaluate all runs on test data; generate all requested metrics and plots.

Everything is contained in these notebooks. They save intermediate data in `artifacts/` and results in `results/`, beside the notebooks. There are no helper scripts to manage. Training notebooks can run in either order after this one. The code uses PyTorch and Gensim.

## What can be reproduced

Source: *Trolls, Haters, or Teammates: Classifying Sentiment in the Language of PvP Game Communications*, Kevin Borst, Thomas Yang, Melinda Zhu, June 3, 2021 (the supplied Base Report.pdf).

- Paper labels (p. 1): **0 = neutral/positive**, **1 = negative attitude**, **2 = derogatory**.
- The supplied files contain 21,120 training, 5,280 validation, and 6,600 test messages. This matches the paper's 64%/16%/20% proportions and 33,000 total, but each supplied split has equal class counts; the paper's Figure 2 is uneven. We preserve your files without re-splitting or rebalancing.
- The paper uses pretrained news embeddings (Section 3.2). Its claimed Google News Word2Vec 50d/100d source is ambiguous: the public Google News release is **300d**. Here the default is to project its vectors to 50d/100d using PCA fitted only on words in the training vocabulary. This is a **documented approximation**, not an exact recovery of the paper's embedding method. Both dimensions use one common PCA basis. An optional `train` mode learns actual 50d/100d Word2Vec on training messages; that is a separate domain-specific experiment, not the paper's pretrained setup.
- GloVe uses the original Wikipedia 2014 + Gigaword 5 vectors, not the newer 2024 release.
- The paper omits tokenization, pooling, hidden widths, batch size, optimizer settings, and whether embeddings are updated. Those choices are stated explicitly in these notebooks. Matching its numerical results is not guaranteed.
- Paper epoch budgets are 550 for feedforward and 300 for LSTM. We use those maximums, with optional early stopping enabled for practicality. Disable early stopping in notebooks 2 and 3 to run the full budgets.

The first pretrained run downloads about **1.95 GB** (mostly Google News) and needs additional disk space for temporary download files. Downloads are cached. The loader reads only the training vocabulary into memory. Internet is needed for installation and first downloads. You may instead supply local embedding paths in the configuration cell.

Official references: [Gensim model catalog](https://github.com/piskvorky/gensim-data), [Gensim downloads](https://radimrehurek.com/gensim/auto_examples/howtos/run_downloader_api.html), [Stanford GloVe](https://nlp.stanford.edu/projects/glove/), [Word2Vec project archive](https://code.google.com/archive/p/word2vec/).


In [ ]:
# Run once in this Python environment. Restart the kernel if imports request it.
%pip install "numpy>=1.26,<3" "pandas>=2.2,<4" "scikit-learn>=1.4,<2" "matplotlib>=3.9,<4" "torch>=2.2,<3" "gensim>=4.4,<5" "certifi>=2024.2"


In [ ]:
from pathlib import Path
from collections import Counter
from itertools import chain
import gzip
import hashlib
import json
import os
import platform
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.decomposition import PCA

# All notebooks must use this same working folder.
ROOT = Path.cwd()
DATA_DIR = Path("/Users/arvind/Documents/Sem5/ML/Mini Project")
# If you move the notebooks, put the three CSVs in a data/ folder beside them.
if not (DATA_DIR / "train.csv").exists():
    DATA_DIR = ROOT / "data"

ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
CACHE = ROOT / "embedding_downloads"
CACHE.mkdir(exist_ok=True)
os.environ["GENSIM_DATA_DIR"] = str(CACHE)
# Set the cache location before importing gensim.downloader.
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec

SEED = 42
WORD2VEC_SOURCE = "google_news_pca"  # or "train" for a separate experiment
GOOGLE_NEWS_FILE = None  # optional local GoogleNews .bin or .bin.gz
GLOVE_FILES = {50: None, 100: None}  # optional local original GloVe .txt files
assert WORD2VEC_SOURCE in {"google_news_pca", "train"}
CLASS_NAMES = ["neutral/positive", "negative attitude", "derogatory"]
print("Notebook folder:", ROOT)
print("CSV folder:", DATA_DIR)


## Inspect the fixed splits

We check missing values, label validity, repeated messages, and normalized text overlap between splits. Overlap can make evaluation optimistic; we report it and retain the supplied split. The labels and texts in the test file are inspected only for this fixed data audit and are not used for fitting or model selection.


In [ ]:
frames = {}
file_hashes = {}
for split in ["train", "validation", "test"]:
    path = DATA_DIR / f"{split}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Set DATA_DIR above.")
    df = pd.read_csv(path)
    assert {"comment_text", "label"}.issubset(df.columns)
    assert df[["comment_text", "label"]].notna().all().all(), f"Missing values: {split}"
    assert df["label"].isin([0, 1, 2]).all(), f"Unexpected labels: {split}"
    df["comment_text"] = df["comment_text"].astype(str)
    df["label"] = df["label"].astype(int)
    frames[split] = df
    file_hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()

counts = pd.DataFrame({name: df["label"].value_counts().reindex([0, 1, 2], fill_value=0)
                       for name, df in frames.items()}).T
counts.columns = CLASS_NAMES
counts["total"] = counts.sum(axis=1)
display(counts)
counts[CLASS_NAMES].plot.bar(rot=0, title="Class counts in the supplied splits")
plt.ylabel("Messages")
plt.tight_layout()
plt.show()


## Tokenize and fit the vocabulary on training messages

Lowercase text; keep words, digits, contractions, and individual punctuation symbols. Do not remove stopwords, censor insults, stem words, or discard game slang. Reserve index 0 for padding and 1 for unknown words. All training words are retained; validation/test words cannot expand the vocabulary. Sequence length is the longest training message, capped at 128 tokens; longer evaluation messages are truncated on the right. Empty messages become one unknown token.


In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?|[^\w\s]", text.lower())

tokens = {split: [tokenize(text) for text in df["comment_text"]]
          for split, df in frames.items()}
normalized = {split: [" ".join(row) for row in rows] for split, rows in tokens.items()}
audit = []
for split, texts in normalized.items():
    audit.append({"check": f"{split}: repeated normalized messages",
                  "count": len(texts) - len(set(texts))})
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    audit.append({"check": f"normalized text overlap: {left} / {right}",
                  "count": len(set(normalized[left]) & set(normalized[right]))})
display(pd.DataFrame(audit))
all_rows = pd.concat([pd.DataFrame({"text": normalized[s], "label": frames[s]["label"]})
                      for s in frames], ignore_index=True)
print("Normalized texts with conflicting labels:",
      int(all_rows.groupby("text")["label"].nunique().gt(1).sum()))

word_counts = Counter(word for row in tokens["train"] for word in row)
# Sorting gives reproducible token IDs, including ties in frequency.
words = sorted(word_counts, key=lambda w: (-word_counts[w], w))
vocab = {"<PAD>": 0, "<UNK>": 1}
vocab.update({word: index + 2 for index, word in enumerate(words)})
MAX_LENGTH = max(1, min(128, max(map(len, tokens["train"]))))

def encode(rows):
    array = np.zeros((len(rows), MAX_LENGTH), dtype=np.int64)
    for i, row in enumerate(rows):
        ids = [vocab.get(word, 1) for word in row[:MAX_LENGTH]] or [1]
        array[i, :len(ids)] = ids
    return array

arrays = {}
for split, rows in tokens.items():
    arrays[f"X_{split}"] = encode(rows)
    arrays[f"y_{split}"] = frames[split]["label"].to_numpy(dtype=np.int64)
    total = sum(map(len, rows))
    unknown = sum(word not in vocab for row in rows for word in row)
    truncated = sum(len(row) > MAX_LENGTH for row in rows)
    print(f"{split}: {len(rows):,} rows; unknown tokens {unknown / max(total, 1):.2%}; "
          f"truncated messages {truncated}")
print("Vocabulary size:", len(vocab), "| Maximum length:", MAX_LENGTH)


## Read pretrained vectors without loading the full model

Gensim downloads and checks the cached files. `read_vectors` handles Google News binary files and GloVe text files (with or without a word2vec header). Only words in the training vocabulary are kept. If a download fails, rerun the cell after fixing the connection or supply a local file; there is no silent replacement with random embeddings.


In [ ]:
def read_vectors(path, wanted, dimension, binary=False):
    path = Path(path)
    opener = gzip.open if path.suffix == ".gz" else open
    found = {}
    with opener(path, "rb") as stream:
        first_line = stream.readline()
        if binary:
            count, width = map(int, first_line.split())
            assert width == dimension
            for _ in range(count):
                word_bytes = bytearray()
                char = stream.read(1)
                while char != b" ":
                    if not char:
                        raise ValueError("Unexpected end of binary embedding file")
                    if char != b"\n":
                        word_bytes.extend(char)
                    char = stream.read(1)
                raw = stream.read(4 * dimension)
                if len(raw) != 4 * dimension:
                    raise ValueError("Incomplete vector in binary embedding file")
                word = word_bytes.decode("utf-8", errors="replace")
                if word in wanted:
                    found[word] = np.frombuffer(raw, dtype="<f4").copy()
        else:
            parts = first_line.split()
            has_header = len(parts) == 2 and all(part.isdigit() for part in parts)
            if has_header:
                assert int(parts[1]) == dimension
            lines = stream if has_header else chain([first_line], stream)
            for line in lines:
                word_bytes, _, raw = line.rstrip().partition(b" ")
                word = word_bytes.decode("utf-8", errors="replace")
                if word in wanted:
                    vector = np.fromstring(raw.decode("utf-8"), sep=" ", dtype=np.float32)
                    assert len(vector) == dimension, f"Bad vector: {word}"
                    found[word] = vector
    if not found:
        raise ValueError(f"No vocabulary words found in {path}")
    assert all(np.isfinite(v).all() for v in found.values())
    return found

# Missing pretrained words get seeded random vectors. <PAD> stays zero;
# <UNK> gets a random vector. Coverage below excludes these random rows.
def make_matrix(vectors, dimension):
    rng = np.random.default_rng(SEED)
    matrix = rng.normal(0, 0.05, (len(vocab), dimension)).astype(np.float32)
    matrix[0] = 0
    for word, vector in vectors.items():
        matrix[vocab[word]] = vector
    return matrix


In [ ]:
embedding_matrices = {}
matched_words = {}
embedding_sources = {}
wanted = set(words)

if WORD2VEC_SOURCE == "google_news_pca":
    path = GOOGLE_NEWS_FILE or api.load("word2vec-google-news-300", return_path=True)
    news_vectors = read_vectors(path, wanted, 300, binary=True)
    matched = sorted(news_vectors)
    values = np.stack([news_vectors[word] for word in matched])
    if min(values.shape) < 100:
        raise ValueError("At least 100 matched training words are needed for PCA to 100d.")
    pca = PCA(n_components=100, svd_solver="full")
    projected = pca.fit_transform(values).astype(np.float32)
    np.savez_compressed(ARTIFACTS / "word2vec_pca_basis.npz",
                        mean=pca.mean_, components=pca.components_)
    print("PCA variance retained:", pca.explained_variance_ratio_.sum())
    for dimension in [50, 100]:
        key = f"word2vec_{dimension}"
        vectors = {word: projected[i, :dimension] for i, word in enumerate(matched)}
        embedding_matrices[key] = make_matrix(vectors, dimension)
        matched_words[key] = set(matched)
        embedding_sources[key] = "Google News 300d -> training-vocabulary PCA"
    del news_vectors, values, projected
else:
    # Separate, explicitly different experiment: skip-gram trained on training data.
    # workers=1 and a stable hash make training repeatable in the same environment.
    def stable_hash(word):
        return int(hashlib.md5(word.encode("utf-8")).hexdigest()[:8], 16)

    for dimension in [50, 100]:
        w2v = Word2Vec(sentences=tokens["train"], vector_size=dimension,
                       window=5, min_count=1, sg=1, negative=5, epochs=30,
                       workers=1, seed=SEED, hashfxn=stable_hash)
        key = f"word2vec_{dimension}"
        vectors = {word: w2v.wv[word] for word in words}
        embedding_matrices[key] = make_matrix(vectors, dimension)
        matched_words[key] = set(vectors)
        embedding_sources[key] = "Skip-gram trained on supplied training messages"

for dimension in [50, 100]:
    path = GLOVE_FILES[dimension] or api.load(f"glove-wiki-gigaword-{dimension}",
                                            return_path=True)
    vectors = read_vectors(path, wanted, dimension)
    key = f"glove_{dimension}"
    embedding_matrices[key] = make_matrix(vectors, dimension)
    matched_words[key] = set(vectors)
    embedding_sources[key] = "GloVe Wikipedia 2014 + Gigaword 5 (6B)"


## Coverage and saved inputs

Type coverage counts distinct training words. Token coverage weights them by training frequency. Validation/test coverage is also reported against the actual pretrained matches, so unknown-word random rows are not counted as pretrained coverage. When using `train` mode, coverage instead means words seen by the locally trained Word2Vec model.

A fingerprint ties every trained model to these data and embedding matrices. If you change preparation settings, rerun both training notebooks; notebook 4 rejects models from a different preparation.


In [ ]:
coverage = []
for key, matrix in embedding_matrices.items():
    dimension = int(key.rsplit("_", 1)[1])
    assert matrix.shape == (len(vocab), dimension)
    assert np.isfinite(matrix).all() and np.all(matrix[0] == 0)
    matched = matched_words[key]
    row = {"embedding": key, "source": embedding_sources[key],
           "training_type_coverage": len(matched) / max(len(words), 1)}
    for split, rows in tokens.items():
        total = sum(map(len, rows))
        covered = sum(word in matched for sentence in rows for word in sentence)
        row[f"{split}_token_coverage"] = covered / max(total, 1)
    coverage.append(row)
coverage_df = pd.DataFrame(coverage)
display(coverage_df)

metadata = {"seed": SEED, "class_names": CLASS_NAMES, "max_length": MAX_LENGTH,
            "vocab_size": len(vocab), "word2vec_source": WORD2VEC_SOURCE,
            "embedding_sources": embedding_sources, "csv_sha256": file_hashes,
            "tokenization": "lowercase; words, contractions, digits, punctuation",
            "split_sizes": {s: len(df) for s, df in frames.items()},
            "audit": audit,
            "versions": {"python": platform.python_version(), "numpy": np.__version__,
                         "pandas": pd.__version__, "gensim": gensim.__version__}}
fingerprint = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode())
for key in sorted(embedding_matrices):
    fingerprint.update(embedding_matrices[key].tobytes())
for key in sorted(arrays):
    fingerprint.update(arrays[key].tobytes())
metadata["preparation_id"] = fingerprint.hexdigest()
np.savez_compressed(ARTIFACTS / "encoded_data.npz", **arrays)
np.savez_compressed(ARTIFACTS / "embeddings.npz", **embedding_matrices)
(ARTIFACTS / "metadata.json").write_text(json.dumps(metadata, indent=2))
(ARTIFACTS / "vocabulary.json").write_text(json.dumps(vocab, indent=2))
coverage_df.to_csv(ARTIFACTS / "embedding_coverage.csv", index=False)
print("Saved four embeddings and encoded splits in", ARTIFACTS)
print("Next: run notebook 02 and notebook 03.")
